## Create a 2D netCDF file 

A flexible notebook to convert data files into netCDF format for upload to Datalakes. This is for 2D data (e.g. temperature at various depths measured over time -- dimension1 = depth, dimension2 = time). 

If you are not sure how many dimensions your data has, please refer to the netCDF_README file.

This python notebook is designed to be flexible to handle various different types of data. Thus it requires user entry into a few files to provide information about the data. 

In [1]:
# import the needed libraries
import os
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib as plt
from pathlib import Path
import yaml
import csv
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
from collections import defaultdict
import json

In [3]:
# import the functions that exist in the folder
from utils import find_file, extract_identifier_from_pattern, sanitize_name, combine_date_time, files_to_process

__________

### Step 1: what format is your data?

**NOTE:** We are assuming that one of your dimensions is time. The other dimension (we will call dimension1) could be a variety of different things; common ones on Datalakes are depth or water pressure.


Let's collect some information about your data files. 

* Does each file only contain one dimension1 value? (e.g. file_1 is measurements of variables at a specific depth, there can be multiple files with the same depth - ex. file_1m_datecollected, file_1m_datecollected2 ...). Then we want to use OPTION A. 

* If a single file has multiple dimension1 values (e.g. file_1 has temperature readings at different depths and depth is your dimension1), use OPTION B. See other script.

* If you have both options, then run the script twice and then you can easily combine your netCDF files. 

In [4]:
# Where is the dimension1 information? Choose ONE:
# Uncomment your choice

# OPTION A: a file does not contain multiple dimension1 values:

# one file = one dimension value (but can have multiple files with the same dimensional value) 
dim_source = 'filename'
# What is the variable name of dimension1? (e.g. depth, water_pressure, ...) do not include spaces
dimension1_name = 'depth'

---------

### Step 2: Find all the data files

Migrate all your data into the data folder: 'data/input/L0'

In [5]:
# the working directory is the folder that contains the full repo:
working_directory = Path.cwd().parent
# the input data directory is where all your data should be:
inputdata_directory = working_directory / 'data/input/L0'
# the script directory is where all the scripts are located:
script_directory = working_directory / 'scripts'

# What is the file naming convention? (need help with this - see README.txt -> understanding components of the code, Step 1)
# if possible, the idvalue should be associated with the dimension value (e.g. the actual value, an instrument number, a serial number ...)
# or some other identification value
data_file_pattern = 'AUTO_{idvalue}_*_data.txt'

In [6]:
### User input not required in this cell, if naming in the above cell has not changed ###

# find all the data files you have put in the data folder (L0):
file_list, file_summary = find_file(inputdata_directory, data_file_pattern)  

# create a metadata file about the data files
dfile_metadata = pd.DataFrame(list(file_summary['files_per_identifier'].items()), columns = ['ID_value', 'number_of_files'])
# add the dimension variable
dimension1_name = sanitize_name(dimension1_name) #remove any spaces, replace with underscores for ease of handling
dfile_metadata[dimension1_name] = ""

# save this info to a csv file
dfile_output_file = inputdata_directory.parent / 'datafile_information.csv'
dfile_metadata.to_csv(dfile_output_file, index = False)

Total files found: 19
Total number of unqiue 'idvalues' found: 19


--------

### Step 3: edit the datafile_information file

In your data/input folder there is now a file labeled 'datafile_information.csv' which has a list of the unique id_values, the number of files per id_value (e.g. how many time steps), and a blank column with your dimension1_name. 

**Check the information in this file and enter the dimension1 values for each id_value.**

If there are id_values you do not wish to process, simply remove the row from this file. We will be using this file to process the data files. 

__________

### Step 4: information about the format of your data files

Please fill in the below with the needed information on the format of your data file:

In [20]:
datafile_attributes = {   
            # Data collection time period (YYYY,mm,dd,hh,min,ss):
            "start_time": datetime(2025,1,17,0,0,0),
            "end_time": datetime(2025,4,22,0,0,0),

            # Do your data files have headers? How many lines of header/text? We will skip those to get to the data.
            "skiprows": 1,

            # What is the separator in your data file (e.g. comma = ',', tab = '\t', space = ' ', semicolon = ';', etc)?
            "separator": ',',
    
            # What is the fill value/missing value number? (e.g. '999', 'NaN', 'NAN') 
            "_FillValue": ' ',

            # What are your date/time column(s) named in your data file? 
            # Ordered from largest to smallest (e.g. Date, Time OR Year, Month, Day, etc.) NOT in the order they are in your datafile
            "time_columns": ['Time'], 

            # Do your data files collect data in UTC time? Enter the time shift in hours or 0 if they are already in UTC:
            "time_shift": 0, 

            # What are the variable names in the data files, including the names of the time/date columns? 
            # We will pull the variables in your file based on these names (ensure they are labelled the same as in the data files)
            # Only include variables you are interested in processing:
            "var_names": ['Time', 'Temperature']
        }

In [29]:
### User input not required in this cell ###
### This will create/overwrite the variables.csv file if you already have one ###

# Convert start and end time to seconds since 1970 for netcdf format
sec_start_time = datafile_attributes["start_time"].timestamp()
sec_end_time = datafile_attributes["end_time"].timestamp() 

# Let's just add the dimension1 name into the datafile_attributes dictionary for later
datafile_attributes["dimension1_name"] = dimension1_name

# Create a variable mapping file:
var_output = inputdata_directory.parent / 'variables.csv'

# Do not edit the below:
# Column names in this file
headers = ['var_name', 'dim', 'unit', 'long_name', 'min', 'max']

# add in dimension1 to this file
dim1_metadata = {'var_name': dimension1_name, 
                 'dim': f'({dimension1_name},)',
                 'unit': '',
                 'long_name': '',
                 'min': '',
                 'max': ''}

# input the time metadata that we already know
time_metadata = {'var_name':'time',
                 'dim': '(time,)',     #var is dependent on this variable
                 'unit': 'seconds since 1970-01-01 00:00:00',
                 'long_name': 'time',
                 'min': sec_start_time - 86400,
                 'max': sec_end_time + 86400}

# let's make the variable file
with open(var_output, mode = 'w', newline='') as file:
     writer = csv.DictWriter(file, fieldnames=headers)
     writer.writeheader()
    
     # write time 
     writer.writerow(time_metadata)
     # write the other dimension in
     writer.writerow(dim1_metadata)
    
     # manage the overlapping time entries if they exist
     list3 = [item for item in var_names if item not in time_columns]
    
     # write in the other variables
     for name in list3:
         #remove any spaces as needed
         name = sanitize_name(name)
         row = {
             'var_name': name.lower(),
             'dim':'',
             'unit': '',
             'long_name': '',
             'min':'',
             'max': ''}
         writer.writerow(row)

_____________

### Step 5: edit the variable file

In your data/input folder there is now a file labeled 'variables.csv'. This information is needed for the netCDF metadata. 

**Check the information in this file and fill in the blank columns for your additional variables. Do not edit the time row.**

Info about this file:
* var_name = shorthand name for the variable (ensure there are no spaces in the var_name)
* dim = the dimension(s) this is variable is dependent on, set up as (dim1, dim2, ...) (see README.md Step 5 for more information, if you are uncertain about this)
* unit = the units (e.g. temp = degrees C etc)
* long_name = the full name for the variable (can be the same as var_name). See CF conventions, linked below, if uncertain.
* min = expected minimum value
* max = expected maximum value


CF naming conventions: [ https://cfconventions.org/Data/cf-standard-names/27/build/cf-standard-name-table.html ]

CF guidelines for construction of names: [ https://cfconventions.org/Data/cf-standard-names/docs/guidelines.html ]

---------------------------

### Step 6: Metadata

For this, you can enter the metadata here manually or use the metadata generator script. 

*Recommended*: The metadata generator can be found on Renku, ready to use, here: [https://renkulab.io/p/ellen.knappe/datalakes-streamlit]. This will create a JSON file in the DataCite format, and included in your data upload. Alterntaively, you can run it on your local machine by forking the repo from the Eawag Gitlab [ https://gitlab.eawag.ch/ellen.knappe/datalakes_streamlit ] or Github [ https://github.com/eknappe/metadata_generator ]. We recommend using the DataCite format because this format is interoperable, machine-readable, and recommended by the ETH Domain.

We will be using ACDD conventions for the netCDF. See here for more info: [ https://wiki.esipfed.org/Attribute_Convention_for_Data_Discovery_1-3 ].

Once you generate the metadata file, put it in the working_directory.

In [8]:
# we recommend putting it the top level directory, e.g. the working_directory that was set at the start of this notebook 
# else define it here
file_location = working_directory
print(f"This is where you should put your metadata file: {working_directory}")

This is where you should put your metadata file: /Users/knappeel/Documents/coding/datalakes/data_convert


**No metadata file? Input information here. If you have a metadata file, skip this cell.** 

In [ ]:
## SKIP THIS SECTION IF YOU CREATED A METADATA FILE ##

# else, input the required fields
general_attributes = {
            "title": " ",
            "creator_name": " ",
            "institution": " ",
            "keywords": "[keyword1,keyword2,..] ",
            "Conventions": "ACDD-1.3",
            "description": " ", #abstract/description of your data
            "source": " ", #add in the name of the lakes here and any other project info needed
            "license": " "
        }

__What is the name of your metadata file?:__

In [9]:
## SKIP THIS CELL AND THE NEXT CELL IF YOU ADDED METADATA MANUALLY ##

# read in the json metadata file
jfilename = 'test_thermistor_data_metadata_20260105_081338.json'

jfilename = file_location / jfilename

In [10]:
## User input not required in this cell ##

# Read in the metadata file provided
with open(jfilename, mode = 'r', encoding = "utf-8") as read_file:
    metadata_json = json.load(read_file)

# extract the needed information from the json metadata file 
title = metadata_json['data']['attributes']['titles'][0]['title']
creator_name = metadata_json['data']['attributes']['creators'][0]['name']
institution = metadata_json['data']['attributes']['creators'][0]['affiliation'][0]['name']
keywords = ', '.join([key['subject'] for key in metadata_json['data']['attributes']['subjects']]) #extracts the keywords if there are multiple
description = metadata_json['data']['attributes']['descriptions'][0]['description']
source =', '.join([geo['geoLocationPlace'] for geo in metadata_json['data']['attributes']['geoLocations']]) #extracts the location if there are multiple
license = metadata_json['data']['attributes']['rightsList'][0]['rights']

# make a dictionary for ease of adding it to the netCDF file
general_attributes = {
            "title": title,
            "creator_name": creator_name,
            "institution": institution,
            "keywords": keywords,
            "Conventions": "ACDD-1.3",
            "description": description,
            "source": source, 
            "license": license
        }

print(general_attributes)

{'title': 'Test: Thermistor data from Jemina', 'creator_name': 'Ellen Knappe', 'institution': 'Lib4RI - Library for the Research Institutes within the ETH Domain: Eawag\\, Empa\\, PSI & WSL', 'keywords': 'temperature, depth', 'Conventions': 'ACDD-1.3', 'description': 'This is some test data kindly provided by Jemima to test this all out. ', 'source': 'Lake Zug', 'license': 'CC BY 4.0'}


______________

### Step 7: read in all the data files

It's show time. 

In [23]:
# no user input needed for this cell #

# first we will cycle through all the files, based on the datafile_information.csv file
# open them, based on the information provided, then combine them into a netcdf 
files_information = pd.read_csv(dfile_output_file)

# convert into a list for the function
file_identifiers = files_information['ID_value'].astype(str).tolist()

# get all the file locations (e.g. the path on your computer):
files_locations = files_to_process(inputdata_directory, data_file_pattern, file_identifiers)

#just a little double check here
print(f"Number of files to process: {len(files_locations)}")
print(f"Number of unique {datafile_attributes["dimension1_name"]}s to be processed: {len(file_identifiers)}")


Number of files to process: 19
Number of unique depths to be processed: 19


In [ ]:
def build_netcdf(files_locations, files_information, dimension1_name, general_attributes, variables_files):
    """
    Build a 2D netcdf from multiple CSV files using a variables definition file. 
    Creates one netCDF file per day with format: L1_projectname_version_YYYYMMDD_000000.nc

    Assumptions:
    - each file only has a singular dimension1 value (e.g. depth) which is identified in files_information
    - all files have the same time values
    - CSV structure is defined above
    - variables_files defines the structure and metadata for the netCDFs

    args:
        files_locations: list of file paths 
        files_information: dataframe with ID_value and their associated dimension1 value
        dimension1_name: name of the first dimension
        general_attributes: dictionary of netCDF global attributes
    """
    
    # ensure output directory exists
    
    
        

In [65]:
# loop through all the files listed

for file in files_locations:
    print(file)
    data = pd.read_csv(file, sep = datafile_attributes["separator"], skiprows = datafile_attributes["skiprows"]-1, usecols = datafile_attributes["var_names"])

print(data)


/Users/knappeel/Documents/coding/datalakes/data_convert/data/input/L0/AUTO_209297_20250424_1557/AUTO_209297_20250424_1557_data.txt
/Users/knappeel/Documents/coding/datalakes/data_convert/data/input/L0/AUTO_205534_20250424_1553/AUTO_205534_20250424_1553_data.txt
/Users/knappeel/Documents/coding/datalakes/data_convert/data/input/L0/AUTO_209290_20250424_1551/AUTO_209290_20250424_1551_data.txt
/Users/knappeel/Documents/coding/datalakes/data_convert/data/input/L0/AUTO_205524_20250424_1555/AUTO_205524_20250424_1555_data.txt
/Users/knappeel/Documents/coding/datalakes/data_convert/data/input/L0/AUTO_209291_20250424_1547/AUTO_209291_20250424_1547_data.txt
/Users/knappeel/Documents/coding/datalakes/data_convert/data/input/L0/AUTO_203319_20250424_1558/AUTO_203319_20250424_1558_data.txt
/Users/knappeel/Documents/coding/datalakes/data_convert/data/input/L0/AUTO_205537_20250424_1549/AUTO_205537_20250424_1549_data.txt
/Users/knappeel/Documents/coding/datalakes/data_convert/data/input/L0/AUTO_209292_2